In [48]:
import os
import pandas as pd
import numpy as np
from pyproj import Transformer
from tqdm import tqdm

from IPython.display import display

In [21]:
ROOT_PATH = "dataset/scale_2_landscape/"

fp_lucas_train_val = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train_val-0.06min.csv")
fp_lucas_test = os.path.join(ROOT_PATH, "glc24_pa_test_private_CBN-med_matching-LUCAS-500m.csv")

df_lucas_train_val = pd.read_csv(fp_lucas_train_val)
df_lucas_test = pd.read_csv(fp_lucas_test)

print(f"LUCAS occurrences Train-Val: {df_lucas_train_val.shape} ({df_lucas_train_val['id'].nunique()} sites)")
display(df_lucas_train_val.head(1))
print(f"\nLUCAS occurrences Test: {df_lucas_test.shape} ({df_lucas_test['id'].nunique()} sites)")
display(df_lucas_test.head(1))

print("\nExcluding Test surveyIds from Train-val...")
df_lucas_train_val = df_lucas_train_val[~(df_lucas_train_val['id'].isin(df_lucas_test['id'].unique()))]
print(f"LUCAS occurrences Train-Val (after purge of Test): {df_lucas_test.shape} ({df_lucas_test['id'].nunique()} sites)")

LUCAS occurrences Train-Val: (47258, 12) (7903 sites)


,Unnamed: 0,id,file_path,full_path,image_source,exists,full_path_missing,full_path_2022,full_path_cover,lon,lat,subset
0,0,967805,2009/FR/378/823/37882398N.jpg,LUCAS/2009/FR/378/823/37882398N.jpg,file_path_gisco_north,False,LUCAS/../../lucas_missing/2009/FR/378/823/3788...,LUCAS/../../lucas_photos_all_2022/2009/FR/378/...,LUCAS/../../lucas_cover/2009/FR/378/823/378823...,3.303906,44.47791,train



LUCAS occurrences Test: (1252, 27) (64 sites)


,PlotObservationID_eva,lon,lat,year,datasetName,Access.regime,Expert.System,Cover.abundance.scale,geoUncertaintyInM,observer,...,region,surveyID,country,speciesId,surveyId,split,id,x_EPSG_32631,y_EPSG_32631,lucas_matching_ids
0,1953693.0,6.215064,43.12536,2021,CBNMed,2,V32,Braun/Blanquet (old),3.0,NaN,...,MEDITERRANEAN,93212,France,10822.0,74414,train,74414,761530.363056,4.779755e+06,751990



Excluding Test surveyIds from Train-val...
LUCAS occurrences Train-Val (after purge of Test): (1252, 27) (64 sites)


In [56]:
def delta_after_shift(d, lon, lat):
    """Computes the delta between 2 WGS84 coords based on a metric distance, through the EPSG:3035.
    
    WARNING: extremely slow to call within a loop.
    
    d: distance in meters (added to both x and y in EPSG:3035)
    lon, lat: original WGS84 coordinates (degrees)

    Returns (delta_lon, delta_lat) between before and after.
    """
    wgs84 = "EPSG:4326"    # lon/lat in WGS84
    epsg3035 = "EPSG:3035" # ETRS89 / LAEA Europe

    # always_xy=True → inputs/outputs are (lon, lat) for geographic CRS
    to_3035 = Transformer.from_crs(wgs84, epsg3035, always_xy=True)
    to_4326 = Transformer.from_crs(epsg3035, wgs84, always_xy=True)

    # 1) WGS84 → EPSG:3035
    x0, y0 = to_3035.transform(lon, lat)

    # 2) Add distance d to each projected coordinate
    x1 = x0 + d
    y1 = y0 + d

    # 3) Back to WGS84
    lon1, lat1 = to_4326.transform(x1, y1)

    # 4) Delta between before and after
    dlon = lon1 - lon
    dlat = lat1 - lat

    return dlon, dlat

def apply_noise(df: pd.DataFrame,
                columns: list[str],
                gamma: float = 0.1,
                epsilon: float = 0.5,
                noise_type: str = "uniform",
                random_state: int = None
               ):
    """
    Apply random noise to a pandas column.

    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): Column name to modify.
        gamma (float): Probability of applying noise to each value (0 to 1).
        epsilon (float): Noise magnitude.
        noise_type (str): Type of noise to apply ("gaussian" or "uniform").
        random_state (int, optional): Seed for reproducibility.

    Returns:
        pd.DataFrame: New dataframe with the noisy column.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df_noisy = df.copy()

    # Decide which rows get noise
    mask = np.random.rand(len(df)) < gamma

    # Generate noise
    if noise_type == "gaussian":
        noise = np.random.normal(loc=0, scale=epsilon, size=len(df))
    elif noise_type == "uniform":
        noise = np.random.uniform(low=epsilon, high=5*epsilon, size=len(df))
    else:
        raise ValueError("noise_type must be 'gaussian' or 'uniform'")
    signs = np.random.choice([-1, 1], size=len(df))

    # Apply noise only where mask is True
    for column in columns:
        df_noisy.loc[mask, column] += signs[mask] * noise[mask]
        df_noisy[f'{column}_original'] = df[column]

    return df_noisy

def apply_noise_over_GPS_in_meters(
    df: pd.DataFrame,
    gamma: float = 0.1,
    epsilon: int = 100,
    random_state: int = None
):
    """
    Apply random noise to a pandas column.

    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): Column name to modify.
        gamma (float): Probability of applying noise to each value (0 to 1).
        epsilon (float): Noise magnitude.
        noise_type (str): Type of noise to apply ("gaussian" or "uniform").
        random_state (int, optional): Seed for reproducibility.

    Returns:
        pd.DataFrame: New dataframe with the noisy column.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df_noisy = df.copy()
    
    wgs84 = "EPSG:4326"    # lon/lat in WGS84
    epsg3035 = "EPSG:3035" # ETRS89 / LAEA Europe
    # always_xy=True → inputs/outputs are (lon, lat) for geographic CRS
    to_3035 = Transformer.from_crs(wgs84, epsg3035, always_xy=True)
    to_4326 = Transformer.from_crs(epsg3035, wgs84, always_xy=True)

    # Decide which rows get noise
    mask = np.random.rand(len(df)) < gamma
    signs_lon = np.random.choice([-1, 1], size=len(df))
    signs_lat = np.random.choice([-1, 1], size=len(df))

    # Apply noise only where mask is True
    df_noisy['lon_original'] = df['lon']
    df_noisy['lat_original'] = df['lat']
    for (rowi, row), maski in tqdm(zip(df.iterrows(), range(len(mask))), total=(len(df))):
        if maski:
            lon = row['lon']
            lat = row['lat']
            x0, y0 = to_3035.transform(lon, lat)
            # 2) Add distance d to each projected coordinate
            x1 = x0 + epsilon
            y1 = y0 + epsilon
            # 3) Back to WGS84
            lon1, lat1 = to_4326.transform(x1, y1)
            # 4) Delta between before and after
            d_lon = lon1 - lon
            d_lat = lat1 - lat

            df_noisy.loc[rowi, 'lon'] += signs_lon[maski] * d_lon
            df_noisy.loc[rowi, 'lat'] += signs_lat[maski] * d_lat
            
    return df_noisy

df_lucas_train_val_noisy_gps = apply_noise_over_GPS_in_meters(df_lucas_train_val, epsilon=100)

100%|███████████████████████████████████| 47258/47258 [00:13<00:00, 3491.99it/s]


In [58]:
df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['lon'] != df_lucas_train_val_noisy_gps['lon_original']].head(2)

,Unnamed: 0,id,file_path,full_path,image_source,exists,full_path_missing,full_path_2022,full_path_cover,lon,lat,subset,lon_original,lat_original
1,1,1144991,2009/FR/375/422/37542270N.jpg,LUCAS/2009/FR/375/422/37542270N.jpg,file_path_gisco_north,False,LUCAS/../../lucas_missing/2009/FR/375/422/3754...,LUCAS/../../lucas_photos_all_2022/2009/FR/375/...,LUCAS/../../lucas_cover/2009/FR/375/422/375422...,3.020948,43.299718,train,3.019830,43.30070
2,2,1112186,2009/FR/395/823/39582304N.jpg,LUCAS/2009/FR/395/823/39582304N.jpg,file_path_gisco_north,False,LUCAS/../../lucas_missing/2009/FR/395/823/3958...,LUCAS/../../lucas_photos_all_2022/2009/FR/395/...,LUCAS/../../lucas_cover/2009/FR/395/823/395823...,5.502324,43.746236,train,5.501157,43.74719


In [59]:
df_lucas_train_val_noisy_gps.to_csv(os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train_val-0.06min_noisy_100m.csv"))

In [60]:
df_lucas_train_val_noisy_gps.shape

(47258, 14)